# DC扫描API示例 - Id-Vg特性测量

使用新的 `DCSweepAPI` 执行 Id-Vg 扫描（转移特性）。

**对比旧方法：**
- ✅ 代码更简洁（从~100行减少到~20行）
- ✅ 自动处理仪器配置和错误
- ✅ 自动生成CSV/JSON/QC文件
- ✅ 可复用的模块化设计

In [ ]:
# 导入
from fefetlab.instruments.visa_session import VisaConfig, VisaSession
from fefetlab.dc import DCSweepAPI

In [ ]:
# 仪器配置
visa_cfg = VisaConfig(
    resource="GPIB0::17::INSTR",
    timeout_ms=30000,
    write_termination="\r\n",
    read_termination="\r\n",
    send_end=True,
)

# 通道映射（根据实际连接调整）
CH_G = 4
CH_D = 5
CH_S = 6

In [ ]:
# 扫描参数
vg_points = [0.0, -0.2, -0.4, -0.6, -0.8, -1.0]  # Vg扫描点
vd_fixed = 0.1   # 固定Vd
vs_fixed = 0.0   # 固定Vs

In [ ]:
# 执行Id-Vg扫描
with VisaSession(visa_cfg) as session:
    # 初始化API
    dc_api = DCSweepAPI(
        session=session,
        ch_g=CH_G,
        ch_d=CH_D,
        ch_s=CH_S
    )
    
    # 运行扫描（自动保存结果）
    result = dc_api.run_idvg_sweep(
        vg_points=vg_points,
        vd_fixed=vd_fixed,
        vs_fixed=vs_fixed,
        auto_export=True,
        verbose=True
    )

# 提取结果
df = result['df']
run_dir = result['run_dir']
qc_df = result['qc_df']

In [ ]:
# 查看测量数据
df[['vg_set', 'vd_set', 'id_A', 'ig_A', 'is_A', 'status']]

In [ ]:
# 查看QC报告
qc_df

In [ ]:
# 简单绘图（如果需要）
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(df['vg_set'], df['id_A'].abs(), 'o-', label='|Id|')
plt.plot(df['vg_set'], df['ig_A'].abs(), 's-', label='|Ig|')
plt.xlabel('Vg (V)')
plt.ylabel('Current (A)')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.legend()
plt.title(f'Id-Vg: Vd={vd_fixed}V, Vs={vs_fixed}V')
plt.tight_layout()
plt.show()

In [ ]:
# 打印保存路径
print(f"\n数据已保存到: {run_dir}")
print(f"  - CSV: {result['data_paths']['csv']}")
print(f"  - JSON: {result['data_paths']['json']}")
print(f"  - QC: {run_dir / 'qc.csv'}")